# R16c step 3 · Step 2's graders with R11s7's pieces: the O1 gate on a second U-Net seed

Task **R16c**, step 3, from `Essential_task_delegation/Progress_Tracker.md` (decided 20 Oct 2026). It was called step 2b until R16c's steps were renumbered 1–5 the same day. Built as a copy of `r16c-step2-stronger-grader.ipynb`, which stays unchanged. Nothing is trained: step 2's two graders and their biases are loaded exactly as they were kept.

**R16's two parts:** the *finding* (SAM 3's buildings on the before-image, plus the U-Net's pieces that SAM 3 missed) and the *grading* (the Siamese grader gives each found building a grade).

**Why**
- Step 2 passed the O1 gate on both grader seeds: SAM 3 + R11's pieces reached Destroyed IoU 0.316 and 0.311, with mIoU 0.587 and 0.585.
- Both seeds used the same U-Net, R11 (seed 42), for the pieces SAM 3 missed and for the reference rows. The gate asks for two seeds, so this run swaps in R11s7: the same recipe, seed 7.

**What changes from step 2**
- The U-Net: R11s7 instead of R11, for the finding's extra pieces, the vote and the reference rows.
- No training: `grader_seed42.pth`, `grader_seed43.pth` and `calibration.json` come from step 2's output. There are no held-out images or crop caches; R16b's fixed rule uses step 2's real shares.
- Two sanity checks:
  - the graders on the true validation buildings must give step 2's macro F1 (0.7779 and 0.7843), since that doesn't depend on the U-Net;
  - R11s7 must reproduce its own `D4_R11s7.csv` (`EXPECTED_UNET` in Cell 2, filled in from that file).

**Expectation, fixed before the run** (20 Oct)
- SAM 3 + R11s7's pieces passes the O1 gate (mIoU above 0.52, Destroyed IoU above 0.30) on both grader seeds, and is kept against R11s7 with TTA by step 2's rule.
- Its scores stay within about 0.01 of step 2's: SAM 3's outlines and the graders are the same, and only the missed pieces change.
- If it fails on either seed, O1's claim stays at one U-Net seed.

**How to run on Kaggle** (account 1, where step 2's and E3's outputs are)
1. Settings: GPU T4 ×1, internet ON, secret `HF_TOKEN` (R11s7's checkpoint is in the private repo `AbrarAlam/disasterm3-d4-R11s7`).
2. Inputs:
   - the DisasterM3 mirror (attached automatically);
   - Add Input → Your Work → `e3-sam3-building-finding`;
   - Add Input → Your Work → `r16c-step2-stronger-grader` (its `R16c-truth` output holds the graders).
3. Run All once with `SMOKE_TEST = True` (8 validation images, a few minutes). Then set it to `False` and use Save & Run All, about 20–30 minutes.
4. Outputs in `/kaggle/working/step3/`:
   - `D4_R16c_step3.csv`, `R16c_step3_summary.csv` (also per source), `grader_true_buildings.csv`;
   - `instances_val.csv`: each found building with both seeds' log-probabilities and its true and R11s7 pixels;
   - `config.json`, `README.md`.

In [1]:
# ── Cell 1: Environment Setup (Run-All safe) ──────────────────────────────
# segmentation-models-pytorch 0.3.4 is needed only to load R11s7 (its checkpoint key layout).
# The grader uses torchvision, which Kaggle already has.
import importlib.metadata as _md
import os, sys, subprocess

PINS = {
    "segmentation-models-pytorch": "0.3.4",   # baseline version (checkpoint key layout)
}

def _mismatched():
    out = []
    for pkg, want in PINS.items():
        try:
            have = _md.version(pkg)
        except _md.PackageNotFoundError:
            have = "not installed"
        if have != want:
            out.append(f"{pkg}: {have} → {want}")
    return out

if _mismatched():
    print("⏳ Installing pinned versions:")
    for _m in _mismatched():
        print(f"   {_m}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *(f"{k}=={v}" for k, v in PINS.items()), "huggingface_hub[hf_xet]"],
                   check=True)                # pip failure stops the run here
    if _mismatched():
        raise RuntimeError(f"Pins still wrong after install: {_mismatched()}")
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
        # background "Save & Run All": nothing imported yet, so new versions load on first import
        print("✓ Dependencies installed (background run) — continuing without restart.")
    else:
        raise SystemExit("✓ Dependencies installed. RESTART THE KERNEL NOW "
                         "(Run → Restart & clear cell outputs), then Run All again.")
else:
    print("✓ All pinned versions already installed — proceeding.")

import torch
print(f"  torch {torch.__version__}, python {sys.version.split()[0]}")

⏳ Installing pinned versions:
   segmentation-models-pytorch: not installed → 0.3.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 38.3 MB/s eta 0:00:00
✓ Dependencies installed (background run) — continuing without restart.
  torch 2.11.0+cu128, python 3.13.15


In [2]:
# ── Cell 2: Configuration and inputs ──────────────────────────────────────
import glob
import json
import os
import random
import re
import time
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import hf_hub_download

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"

NUM_CLASSES = 4                     # 0=Background, 1=Intact, 2=Damaged, 3=Destroyed
CLASS_NAMES = ["Background", "Intact", "Damaged", "Destroyed"]
GRADES = CLASS_NAMES[1:]
LABELS = GRADES                     # step 2's graders learned on the true pieces: no "None" label
SMOKE_TEST = False                  # True = 8 validation images
OUT_DIR = Path("/kaggle/working/step3")

CROP, MARGIN, MIN_AREA = 128, 32, 20     # step 2's crop and margin, R16's smallest piece: as the graders were trained
SEEDS = (42, 43)

# The second U-Net seed for the finding's extra pieces, the vote and the reference rows. Like R11, R11s7 never trained
# on benchmark pictures, so these outputs may answer benchmark questions.
UNET = "R11s7"
EXPECTED_UNET = {"std": (0.5867, 0.2869), "tta": (0.5913, 0.2928)}   # R11s7's own scores, from D4_R11s7.csv
HF_USER = "AbrarAlam"               # checkpoints: {HF_USER}/disasterm3-d4-<run>/final_model.pth
KEEP_RULE = {"min_destroyed_gain": 0.013, "max_miou_drop": 0.005}     # against the U-Net with TTA, as in step 2
STEP2 = {42: (0.5869, 0.3159), 43: (0.5852, 0.3106)}                  # step 2, SAM 3 + R11's pieces: (mIoU, Destroyed IoU)
STEP2_MACRO_F1 = {42: 0.7779, 43: 0.7843}                              # step 2's graders on the true validation buildings

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def find_checkpoint(name):
    """As in E1: a /kaggle/input folder named R11s7 / d4_R11s7 / R11s7-results, then Hugging Face."""
    wanted = {name.lower(), f"d4_{name}".lower(), f"{name}-results".lower()}
    hits = [p for p in glob.glob("/kaggle/input/**/*.pth", recursive=True)
            if any(part.lower() in wanted for part in Path(p).parts)]
    hits.sort(key=lambda p: Path(p).name != "final_model.pth")   # prefer final_model.pth
    if hits:
        return hits[0]
    repo = f"{HF_USER}/disasterm3-d4-{name}"
    errors = []
    for filename in ("final_model.pth", "last.pth"):   # last.pth holds the same epoch-40 weights
        try:
            return hf_hub_download(repo, filename, token=hf_token())
        except Exception as e:
            errors.append(f"{filename}: {type(e).__name__}")
    raise FileNotFoundError(f"No checkpoint for {name}: nothing under /kaggle/input and {repo} gave {errors}. "
                            f"Add a Kaggle dataset with {name}/final_model.pth.")

def find_dir(name, why):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"No '{name}' folder under /kaggle/input: {why}")
    return Path(hits[0])

# The finding on the validation images: E3's SAM 3 maps of the before-image, 2x2 grid (16-bit PNG, 0 = background)
SAM3_DIR = find_dir("pre_grid2", "add E3's output (e3-sam3-building-finding), or a dataset with its masks/pre_grid2/")
GRADER_DIR = find_dir("R16c-truth", "add step 2's output (r16c-step2-stronger-grader): its graders and biases")
assert all((GRADER_DIR / f).exists() for f in ("grader_seed42.pth", "grader_seed43.pth", "calibration.json",
                                               "config.json")), f"{GRADER_DIR} lacks step 2's graders or biases"
UNET_CKPT = find_checkpoint(UNET)

OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"✓ R16c step 3{'  [SMOKE TEST]' if SMOKE_TEST else ''}: step 2's graders (seeds {SEEDS}) with {UNET}'s pieces")
print(f"  Finding (SAM 3): {SAM3_DIR} ({len(list(SAM3_DIR.glob('*.png')))} maps)")
print(f"  Graders and biases: {GRADER_DIR}")
print(f"  U-Net {UNET}: {UNET_CKPT}")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}; outputs: {OUT_DIR}")

final_model.pth: reconstructing file:   0%|          |  0.00B /  130MB            

final_model.pth: downloading bytes:           |  0.00B            

✓ R16c step 3: step 2's graders (seeds (42, 43)) with R11s7's pieces
  Finding (SAM 3): /kaggle/input/notebooks/abrarmohammedtanzim/e3-sam3-building-finding/E3/masks/pre_grid2 (645 maps)
  Graders and biases: /kaggle/input/notebooks/abrarmohammedtanzim/r16c-step2-stronger-grader/R16c-truth
  U-Net R11s7: /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-d4-R11s7/snapshots/d0654a16a2b8b15d8c9989f22519d53cf4d403c0/final_model.pth
  Device: Tesla T4; outputs: /kaggle/working/step3


## 1. Data: the same 645 validation images as every D4 run
Cell 3 is copied verbatim from R11 (via R16 and R16c step 2), so the split matches R1–R16c exactly and every pair keeps its before-image. Only the validation images are used.

In [3]:
# ── Cell 3: Load manifest and build the 4-class masks (verbatim from R11's Cell 4) ──
import cv2
import numpy as np
import os
from collections import Counter, defaultdict
from pathlib import Path
import json

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

# ── Filter: ONLY Building Damage Assessment segmentation entries ──
bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

# ── Path resolution (same logic as before) ──
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

# ── Group entries by underlying post-disaster image ──
by_image = defaultdict(dict)  # post_image_path -> {folder_type: mask_path}

for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":   # skip SAR duplicates, keep optical only
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name  # e.g. train_building_intact_mask
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img
        by_image[post_img]["pre_image_path"] = e.get("pre_image_path")   # R11: the before-image

print(f"Unique images with at least one mask: {len(by_image):,}")

# ── Build combined (image, mask) pairs ──
FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,   # use 2 here instead if you want to collapse into "damaged"
}

# ── FIX: Create temporary directory on Kaggle hard drive for masks ──
os.makedirs("/tmp/combined_masks", exist_ok=True)

pairs = []
skipped_img = 0

for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue

    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id   # later classes overwrite earlier if they overlap

    if combined_mask is None:
        continue  # no valid masks found for this image at all

    # ── FIX: Save mask to hard drive instead of RAM ──
    mask_filename = Path(img_path).name
    save_path = f"/tmp/combined_masks/{mask_filename}"
    cv2.imwrite(save_path, combined_mask)

    pairs.append({
        "image_path": img_path,
        "mask_path": save_path,   # Store the string path, NOT the array!
        "pre_image_path": resolve_path(mask_dict.get("pre_image_path")),   # R11: None if not found
    })

print(f"\n✓ Built {len(pairs):,} valid (image, mask) pairs")
print(f"  Skipped (image not found): {skipped_img:,}")
print(f"  Before-image not found: {sum(p['pre_image_path'] is None for p in pairs):,} (must be 0 for R11)")

# Sanity check
if pairs:
    print(f"\n  Sample: {pairs[0]['image_path']}")
    print(f"  Before: {pairs[0]['pre_image_path']}")
    print(f"  Mask saved at: {pairs[0]['mask_path']}")

Loaded 92,968 entries from manifest
Building Damage Assessment entries: 14,531
Unique images with at least one mask: 6,443

✓ Built 6,443 valid (image, mask) pairs
  Skipped (image not found): 0
  Before-image not found: 0 (must be 0 for R11)

  Sample: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_post_0.png
  Before: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_pre_0.png
  Mask saved at: /tmp/combined_masks/bata_explosion_post_0.png


In [4]:
# ── Cell 4: The validation split (identical to every D4 run) and event names ──
from sklearn.model_selection import train_test_split

train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
if SMOKE_TEST:
    val_pairs = val_pairs[:8]
missing_pre = sum(p["pre_image_path"] is None for p in val_pairs)
assert missing_pre == 0, f"{missing_pre} pairs have no before-image"

XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")
OTHER_NAME = re.compile(r"^(.+?)_post")

def event_and_source(image_path):
    """Same rule as E1: 'hurricane_florence_00000261_post_disaster.png' -> ('hurricane_florence', 'xBD')."""
    name = Path(image_path).name
    m = XBD_NAME.match(name)
    if m:
        return m.group(1), "xBD"
    m = OTHER_NAME.match(name)
    return re.sub(r"\d+$", "", m.group(1) if m else Path(name).stem), "other"

events = [event_and_source(p["image_path"]) for p in val_pairs]
n_xbd = sum(1 for _, source in events if source == "xBD")
print(f"✓ Validation: {len(val_pairs)} images ({n_xbd} xBD, {len(val_pairs) - n_xbd} other). "
      "Expected 645 (560 xBD, 85 other) for the full run.")

✓ Validation: 645 images (560 xBD, 85 other). Expected 645 (560 xBD, 85 other) for the full run.


In [5]:
# ── Cell 5: Step 2's crop helpers, and the true buildings of an image ────────
from scipy import ndimage

def read_triplet(pair):
    """After-image and before-image (RGB, same size) and the 4-class mask, as in the D4 dataset class."""
    post, pre = cv2.imread(pair["image_path"]), cv2.imread(pair["pre_image_path"])
    gt = cv2.imread(pair["mask_path"], cv2.IMREAD_GRAYSCALE)
    if post is None or pre is None or gt is None:
        return None
    post, pre = cv2.cvtColor(post, cv2.COLOR_BGR2RGB), cv2.cvtColor(pre, cv2.COLOR_BGR2RGB)
    h, w = post.shape[:2]
    if pre.shape != post.shape:
        pre = cv2.resize(pre, (w, h), interpolation=cv2.INTER_LINEAR)
    if gt.shape != (h, w):
        gt = cv2.resize(gt, (w, h), interpolation=cv2.INTER_NEAREST)
    return post, pre, gt

def square_box(sl, h, w):
    """Square window around a building's (row, column) slices, MARGIN px wider on each side, kept inside the image."""
    y0, y1, x0, x1 = sl[0].start, sl[0].stop, sl[1].start, sl[1].stop
    side = min(max(y1 - y0, x1 - x0) + 2 * MARGIN, h, w)
    cy, cx = (y0 + y1) / 2, (x0 + x1) / 2
    top = int(round(min(max(cy - side / 2, 0), h - side)))
    left = int(round(min(max(cx - side / 2, 0), w - side)))
    return top, left, side

def building_crops(pre, post, labels, ids):
    """labels: int map of building ids (0 = none). Returns uint8 (len(ids), 7, CROP, CROP):
    before RGB, after RGB, and 255 where the pixel belongs to that building."""
    h, w = labels.shape
    slices = ndimage.find_objects(labels)
    out = np.empty((len(ids), 7, CROP, CROP), np.uint8)
    for n, lid in enumerate(ids):
        top, left, side = square_box(slices[lid - 1], h, w)
        win = (slice(top, top + side), slice(left, left + side))
        out[n, 0:3] = cv2.resize(pre[win], (CROP, CROP), interpolation=cv2.INTER_LINEAR).transpose(2, 0, 1)
        out[n, 3:6] = cv2.resize(post[win], (CROP, CROP), interpolation=cv2.INTER_LINEAR).transpose(2, 0, 1)
        out[n, 6] = cv2.resize((labels[win] == lid).astype(np.uint8) * 255, (CROP, CROP),
                               interpolation=cv2.INTER_NEAREST)
    return out

def pieces(gt):
    """The true buildings of one image as (label, id map, ids, true pixels per building (n, 4)): the 8-connected
    pieces of each true class of at least MIN_AREA px (R16's rule, step 2's "truth" outlines)."""
    out = []
    for cls in (1, 2, 3):
        n, labels, stats, _ = cv2.connectedComponentsWithStats((gt == cls).astype(np.uint8), connectivity=8)
        ids = [i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= MIN_AREA]
        if ids:
            px = np.zeros((len(ids), 4), np.int64)
            px[:, cls] = stats[ids, cv2.CC_STAT_AREA]
            out.append((cls - 1, labels, ids, px))
    return out

# Self-checks (step 2's): a box at the image corner stays inside, each crop marks only its own building, the pieces rule
_lab = np.zeros((200, 300), np.int32)
_lab[0:10, 0:20] = 1
_lab[100:150, 200:260] = 2
assert square_box(ndimage.find_objects(_lab)[0], 200, 300) == (0, 0, 20 + 2 * MARGIN)
_crops = building_crops(np.zeros((200, 300, 3), np.uint8), np.zeros((200, 300, 3), np.uint8), _lab, [1, 2])
assert _crops.shape == (2, 7, CROP, CROP) and _crops[1, 6].max() == 255 and _crops[0, 6].mean() < 255
_gt = np.zeros((200, 300), np.uint8)
_gt[0:10, 0:20], _gt[100:150, 200:260], _gt[190:192, 0:2] = 3, 1, 2      # the 4-pixel piece is below MIN_AREA
assert [(l, len(i)) for l, _, i, _ in pieces(_gt)] == [(0, 1), (2, 1)]
print("✓ Crop helpers ready (self-checks passed)")

✓ Crop helpers ready (self-checks passed)


In [6]:
# ── Cell 6: Step 2's two graders and their biases, loaded as they were kept ──
import torch.nn as nn
import torch.nn.functional as F
import torchvision

class SiameseGrader(nn.Module):
    """Step 2's network: a shared ResNet-18 reads the before and the after crop (RGB + building mask each);
    [f_before, f_after, f_after - f_before] -> one score per label. All its weights come from step 2's files."""

    def __init__(self):
        super().__init__()
        net = torchvision.models.resnet18(weights=None)
        net.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)
        net.fc = nn.Identity()
        self.encoder = net
        self.head = nn.Sequential(nn.Linear(3 * 512, 256), nn.ReLU(inplace=True), nn.Dropout(0.2),
                                  nn.Linear(256, len(LABELS)))

    def forward(self, before, after):
        fb, fa = self.encoder(before), self.encoder(after)
        return self.head(torch.cat([fb, fa, fa - fb], 1))

MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1) * 255
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1) * 255

def to_inputs(x):
    """uint8 (B, 7, S, S) -> normalised before and after tensors (B, 4, S, S); the mask becomes -1 / +1."""
    x = x.to(DEVICE, non_blocking=True).float()
    mask = x[:, 6:7] / 127.5 - 1
    return torch.cat([(x[:, 0:3] - MEAN) / STD, mask], 1), torch.cat([(x[:, 3:6] - MEAN) / STD, mask], 1)

@torch.no_grad()
def log_probs(model, crops):
    """uint8 crops (N, 7, S, S), an array or a memmap -> log-probabilities (N, labels)."""
    model.eval()
    out = []
    for i in range(0, len(crops), 512):
        before, after = to_inputs(torch.from_numpy(np.ascontiguousarray(crops[i:i + 512])))
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            out.append(torch.log_softmax(model(before, after).float(), 1).cpu())
    return torch.cat(out).numpy() if out else np.zeros((0, len(LABELS)), np.float32)

def to_class(labels):
    """Grader labels -> mask classes: intact, damaged, destroyed -> 1-3; "None" -> 0 (background)."""
    return np.where(labels < 3, labels + 1, 0)

step2_cfg = json.loads((GRADER_DIR / "config.json").read_text(encoding="utf-8"))
calibration = json.loads((GRADER_DIR / "calibration.json").read_text(encoding="utf-8"))
assert step2_cfg["labels"] == LABELS and (step2_cfg["crop"], step2_cfg["margin"], step2_cfg["min_area"]) == (CROP, MARGIN, MIN_AREA), \
    "step 2's graders were trained with other labels or crops"
graders, BIASES = {}, {}
for seed in SEEDS:
    g = SiameseGrader().to(DEVICE)
    g.load_state_dict(torch.load(GRADER_DIR / f"grader_seed{seed}.pth", map_location=DEVICE))
    graders[seed] = g.eval()
    BIASES[seed] = np.array([calibration[str(seed)]["bias"][l] for l in LABELS])
REAL_SHARE = np.array([step2_cfg["real_share"][l] for l in LABELS])
PRIOR_BIAS = np.log(REAL_SHARE) - np.log(1 / len(LABELS))      # R16b's fixed rule, as step 2 scored it
PRIOR_BIAS -= PRIOR_BIAS[0]                                     # intact at 0, like the searched biases
print(f"✓ Graders loaded: {', '.join(f'seed {s} (epoch {calibration[str(s)]['epoch']}, biases {calibration[str(s)]['bias']})' for s in SEEDS)}")
print(f"  R16b's fixed rule for reference: {dict(zip(LABELS, PRIOR_BIAS.round(2).tolist()))}")

✓ Graders loaded: seed 42 (epoch 6, biases {'Intact': 0.0, 'Damaged': -1.75, 'Destroyed': -3.75}), seed 43 (epoch 6, biases {'Intact': 0.0, 'Damaged': -2.0, 'Destroyed': -4.0})
  R16b's fixed rule for reference: {'Intact': 0.0, 'Damaged': -1.44, 'Destroyed': -2.13}


## 2. Validation, once: R11s7, the graders on the true buildings, the R16 maps and the ceiling
For each of the 645 validation images (Cells 7 and 8 are step 2's, unchanged):
1. **R11s7** predicts with and without TTA (the D4 code). Its scores must reproduce `D4_R11s7.csv`.
2. **Each grader on the true buildings**, with its own biases. This doesn't depend on the U-Net, so it must give step 2's macro F1.
3. **The finding, as in R16:** SAM 3's instances, plus R11s7's building pieces that SAM 3 missed (at least 20 px).
4. **The ceiling:** a perfect grader on the same outlines.
5. **Each seed's maps:** SAM 3 only, SAM 3 + R11s7's pieces, and the vote with R11s7, with the searched biases and with R16b's fixed rule.

In [7]:
# ── Cell 7: The U-Net (the D4 prediction code) and the E-protocol metrics (step 2's Cell 8) ───
import segmentation_models_pytorch as smp

def load_unet(path):
    """D4 checkpoint: U-Net + ResNet-50, 6 input channels (after-image, then before-image), 4 classes."""
    state = torch.load(path, map_location=DEVICE, weights_only=False)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=6, classes=NUM_CLASSES)
    model.load_state_dict(state)
    return model.to(DEVICE).eval()

IMNET_MEAN = np.array([0.485, 0.456, 0.406], np.float32) * 255
IMNET_STD = np.array([0.229, 0.224, 0.225], np.float32) * 255

def unet_input(post, pre):
    """As R11's val_transform: ImageNet normalisation, channels 0-2 after-image, 3-5 before-image."""
    x = np.concatenate([(post.astype(np.float32) - IMNET_MEAN) / IMNET_STD,
                        (pre.astype(np.float32) - IMNET_MEAN) / IMNET_STD], 2)
    return torch.from_numpy(x.transpose(2, 0, 1).copy()).unsqueeze(0).to(DEVICE)

def _forward_once(model, x):
    """Logits at x's own HxW: pad to a multiple of 32, exactly as in the D4 notebooks."""
    h, w = x.shape[-2:]
    ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
    if ph or pw:
        x = F.pad(x, (0, pw, 0, ph), mode="reflect")
    return model(x)[..., :h, :w].float()

# The D4 notebooks' 8 dihedral views: rotation 0/90/180/270 × (no flip, horizontal flip)
TTA_VIEWS = [(k, flip) for flip in (False, True) for k in range(4)]

def _tta_fwd(x, k, flip):
    x = torch.rot90(x, k, dims=(2, 3))
    return torch.flip(x, dims=(3,)) if flip else x

def _tta_inv(y, k, flip):
    if flip:
        y = torch.flip(y, dims=(3,))
    return torch.rot90(y, -k, dims=(2, 3))

@torch.no_grad()
def predict_std_and_tta(model, x):
    """One pass over the 8 views (as in E1). The identity view (k=0, no flip) is the standard prediction."""
    prob_sum, pred_std = 0, None
    with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
        for k, flip in TTA_VIEWS:
            logits = _forward_once(model, _tta_fwd(x, k, flip))
            if k == 0 and not flip:
                pred_std = logits.argmax(1)
            prob_sum = prob_sum + torch.softmax(_tta_inv(logits, k, flip), dim=1)
    pred_tta = (prob_sum / len(TTA_VIEWS)).argmax(1)
    return (pred_std[0].cpu().numpy().astype(np.uint8),
            pred_tta[0].cpu().numpy().astype(np.uint8))

def confusion(gt, pred):
    """4×4 pixel counts, rows = truth, columns = prediction."""
    idx = gt.astype(np.int64).ravel() * NUM_CLASSES + pred.astype(np.int64).ravel()
    return np.bincount(idx, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)

def metrics(cm):
    """Dataset-level metrics from a confusion matrix (E1's formulas), plus building IoU (classes 1-3 merged)."""
    cm = cm.astype(np.float64)
    eps = 1e-8
    tp = np.diag(cm)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    iou = tp / (tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    building_tp = cm[1:, 1:].sum()
    building_iou = building_tp / (building_tp + cm[0, 1:].sum() + cm[1:, 0].sum() + eps)
    return {"miou": float(iou.mean()), "iou": iou, "precision": precision, "recall": recall,
            "f1": 2 * precision * recall / (precision + recall + eps),
            "f2": 5 * precision * recall / (4 * precision + recall + eps),
            "building_iou": float(building_iou)}

_toy = np.array([[0, 1], [2, 3]], np.uint8)
assert (metrics(confusion(_toy, _toy))["iou"] > 0.99).all()
unet = load_unet(UNET_CKPT)
print(f"✓ {UNET} loaded from {UNET_CKPT} (metric self-check passed)")

✓ R11s7 loaded from /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-d4-R11s7/snapshots/d0654a16a2b8b15d8c9989f22519d53cf4d403c0/final_model.pth (metric self-check passed)


In [8]:
# ── Cell 8: Validation, image by image (step 2's Cell 9) ────────────────────
from tqdm.auto import tqdm

U = UNET.lower()
MAPS = ["sam3", f"sam3+{U}", "vote"]
RULES = {"": "searched biases", "_shares": "R16b's fixed rule"}
VARIANTS = ([f"{UNET}_std", f"{UNET}_tta", "ceiling_sam3", f"ceiling_sam3+{U}", f"ceiling_rejects_sam3+{U}"]
            + [f"R16c_s{seed}_{m}{rule}" for seed in SEEDS for rule in RULES for m in MAPS])
SOURCES = ("all", "xBD", "other")
cms = {v: {s: np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64) for s in SOURCES} for v in VARIANTS}
true_cm = {seed: np.zeros((3, len(LABELS)), np.int64) for seed in SEEDS}   # rows: true grade, columns: label
inst_rows, missing_sam = [], []

t0 = time.time()
for pair, (event, source) in tqdm(list(zip(val_pairs, events)), desc="Validation"):
    data = read_triplet(pair)
    if data is None:
        continue
    post, pre, gt = data
    h, w = gt.shape
    stem = Path(pair["image_path"]).stem

    # 1) The U-Net, standard and 8-view TTA
    pred_std, pred_tta = predict_std_and_tta(unet, unet_input(post, pre))

    # 2) Each grader on the true buildings (object level), with its own biases
    true_parts = pieces(gt)
    if true_parts:
        true_crops = np.concatenate([building_crops(pre, post, lab, ids) for _, lab, ids, _ in true_parts])
        true_labels = np.concatenate([np.full(len(ids), l) for l, _, ids, _ in true_parts])
        for seed, g in graders.items():
            np.add.at(true_cm[seed], (true_labels, (log_probs(g, true_crops) + BIASES[seed]).argmax(1)), 1)

    # 3) The finding, as in R16: SAM 3's instances, then the U-Net's building pieces that SAM 3 missed
    sam = cv2.imread(str(SAM3_DIR / f"{stem}.png"), cv2.IMREAD_UNCHANGED)
    if sam is None:
        missing_sam.append(stem)
        sam = np.zeros((h, w), np.uint16)
    if sam.shape != (h, w):
        sam = cv2.resize(sam, (w, h), interpolation=cv2.INTER_NEAREST)
    ids = sam.astype(np.int32)
    n_sam = int(ids.max())
    n, extra, stats, _ = cv2.connectedComponentsWithStats(((pred_tta > 0) & (ids == 0)).astype(np.uint8),
                                                          connectivity=8)
    keep = stats[:, cv2.CC_STAT_AREA] >= MIN_AREA
    keep[0] = False
    new_id = np.zeros(n, np.int32)
    new_id[keep] = n_sam + 1 + np.arange(int(keep.sum()))
    ids = np.where(extra > 0, new_id[extra], ids)          # ids above n_sam are the U-Net's pieces

    # 4) The ceiling on these outlines, then each seed's grades and its three maps
    present = np.unique(ids)
    present = present[present > 0]
    size = int(ids.max()) + 1
    is_present = np.zeros(size, bool)
    is_present[present] = True
    from_unet = np.arange(size) > n_sam
    flat = ids.ravel().astype(np.int64) * NUM_CLASSES
    unet_px = np.bincount(flat + pred_tta.ravel(), minlength=size * NUM_CLASSES).reshape(size, NUM_CLASSES)
    gt_px = np.bincount(flat + gt.ravel(), minlength=size * NUM_CLASSES).reshape(size, NUM_CLASSES)
    unet_bld = unet_px[:, 1:].astype(np.float32)
    unet_sum = unet_bld.sum(1, keepdims=True)
    perfect = np.where(is_present, gt_px[:, 1:].argmax(1) + 1, 0).astype(np.uint8)   # the main true grade
    rejects = np.where(is_present, gt_px.argmax(1), 0).astype(np.uint8)              # may also say background
    preds = {f"{UNET}_std": pred_std, f"{UNET}_tta": pred_tta,
             "ceiling_sam3": np.where(from_unet, 0, perfect)[ids], f"ceiling_sam3+{U}": perfect[ids],
             f"ceiling_rejects_sam3+{U}": rejects[ids]}
    crops = building_crops(pre, post, ids, present) if len(present) else None
    logps = {}
    for seed, g in graders.items():
        logp = np.zeros((size, len(LABELS)), np.float32)
        if len(present):
            logp[present] = log_probs(g, crops)
        logps[seed] = logp
        for rule, bias in (("", BIASES[seed]), ("_shares", PRIOR_BIAS)):
            scores = logp + bias
            label = scores.argmax(1)
            by_grader = np.where(is_present, to_class(label), 0).astype(np.uint8)
            grade_p = np.exp(scores[:, :3] - scores[:, :3].max(1, keepdims=True))   # the three grades, renormalised
            grade_p /= grade_p.sum(1, keepdims=True)
            shares = np.where(unet_sum > 0, unet_bld / np.maximum(unet_sum, 1), grade_p)   # no U-Net pixel: grader only
            by_vote = np.where(is_present & (label < 3), (0.5 * (grade_p + shares)).argmax(1) + 1, 0).astype(np.uint8)
            preds[f"R16c_s{seed}_sam3{rule}"] = np.where(from_unet, 0, by_grader)[ids]
            preds[f"R16c_s{seed}_sam3+{U}{rule}"] = by_grader[ids]
            preds[f"R16c_s{seed}_vote{rule}"] = by_vote[ids]                   # "None" stays background
    for v, p in preds.items():
        cm = confusion(gt, p)
        cms[v]["all"] += cm
        cms[v][source] += cm

    for lid in present:
        inst_rows.append({"image": f"{stem}.png", "event": event, "source": source, "id": int(lid),
                          "found_by": "sam3" if lid <= n_sam else U, "area_px": int(gt_px[lid].sum()),
                          **{f"logp_{l.lower()}_s{seed}": round(float(logps[seed][lid, j]), 4)
                             for seed in SEEDS for j, l in enumerate(LABELS)},
                          **{f"{U}_{c.lower()}_px": int(unet_px[lid, i]) for i, c in enumerate(CLASS_NAMES)},
                          **{f"gt_{c.lower()}_px": int(gt_px[lid, i]) for i, c in enumerate(CLASS_NAMES)}})

if missing_sam:
    print(f"⚠ {len(missing_sam)} validation images had no SAM 3 map (they got the U-Net's pieces only), "
          f"e.g. {missing_sam[:3]}")
assert len(missing_sam) <= 0.05 * len(val_pairs), "Most SAM 3 maps are missing: is SAM3_DIR E3's pre_grid2 folder?"
VAL_MIN = (time.time() - t0) / 60
print(f"✓ {len(val_pairs)} validation images in {VAL_MIN:.1f} min; {len(inst_rows):,} found buildings graded "
      f"by {len(SEEDS)} grader(s)")

Validation:   0%|          | 0/645 [00:00<?, ?it/s]

✓ 645 validation images in 26.7 min; 58,564 found buildings graded by 2 grader(s)


## 3. Results: the sanity checks, the keep rule and the O1 gate on the second U-Net seed, and the output files

In [9]:
# ── Cell 9: Results, the sanity checks, the keep rule and the gate, and the output files ──
import csv
import pandas as pd

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

# 1) Pixel scores per variant, overall and per source
rows = []
for v in VARIANTS:
    for s in SOURCES:
        m = metrics(cms[v][s])
        rows.append({"variant": v, "source": s, "miou": round(m["miou"], 4),
                     **{f"{c.lower()}_iou": round(float(m["iou"][i]), 4) for i, c in enumerate(CLASS_NAMES)},
                     "building_iou": round(m["building_iou"], 4),
                     "destroyed_precision": round(float(m["precision"][3]), 4),
                     "destroyed_recall": round(float(m["recall"][3]), 4)})
summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "R16c_step3_summary.csv", index=False)
overall = summary[summary["source"] == "all"].drop(columns="source")
print(overall.to_string(index=False))

# 2) Sanity check 1: the U-Net must reproduce its D4 CSV
if SMOKE_TEST:
    sanity = "n/a (smoke test)"
elif EXPECTED_UNET is None:
    sanity = "not checked: fill in EXPECTED_UNET in Cell 2 from D4_R11s7.csv"
else:
    ok = all(abs(metrics(cms[f"{UNET}_{mode}"]["all"])["miou"] - exp[0]) < 5e-4
             and abs(metrics(cms[f"{UNET}_{mode}"]["all"])["iou"][3] - exp[1]) < 5e-4
             for mode, exp in EXPECTED_UNET.items())
    sanity = "yes" if ok else "NO"
print(f"\n{UNET} reproduces its D4 CSV: {sanity}")
if sanity == "NO":
    print("⚠ Check the checkpoint (wrong run or not the last epoch) before trusting the comparison.")

# 3) Sanity check 2: the graders on the true validation buildings (object level, rows = truth) must give step 2's
#    macro F1, since neither the graders nor the true buildings depend on the U-Net
true_rows, macro_f1 = [], {}
for seed in SEEDS:
    cm = true_cm[seed].astype(np.float64)
    tp = np.diag(cm[:, :3])
    precision = tp / np.maximum(cm[:, :3].sum(0), 1)
    recall = tp / np.maximum(cm.sum(1), 1)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-8)
    macro_f1[seed] = round(float(f1.mean()), 4)
    true_rows += [{"seed": seed, "grade": g, "buildings": int(cm[i].sum()), "precision": round(float(precision[i]), 4),
                   "recall": round(float(recall[i]), 4), "f1": round(float(f1[i]), 4),
                   **{f"graded_{l.lower()}": int(true_cm[seed][i, j]) for j, l in enumerate(LABELS)}}
                  for i, g in enumerate(GRADES)]
true_df = pd.DataFrame(true_rows)
true_df.to_csv(OUT_DIR / "grader_true_buildings.csv", index=False)
graders_same = ("n/a (smoke test)" if SMOKE_TEST else
                "yes" if all(abs(macro_f1[s] - STEP2_MACRO_F1[s]) < 1e-3 for s in SEEDS) else "NO")
print(f"\nThe graders on the true validation buildings: macro F1 {macro_f1}; step 2's {STEP2_MACRO_F1}: "
      f"reproduced: {graders_same}")
if graders_same == "NO":
    print("⚠ These aren't step 2's graders as they were kept: check the attached output before trusting the run.")

# 4) The keep rule and the Phase Gate against the U-Net with TTA (searched biases), and the change from step 2
base = metrics(cms[f"{UNET}_tta"]["all"])
verdicts, gates, vs_step2 = {}, {}, {}
print(f"\nKeep rule against {UNET} TTA (Destroyed IoU up by more than {KEEP_RULE['min_destroyed_gain']}, "
      f"mIoU down by at most {KEEP_RULE['max_miou_drop']}):")
for seed in SEEDS:
    for m in MAPS:
        v = f"R16c_s{seed}_{m}"
        mm = metrics(cms[v]["all"])
        d_miou, d_destroyed = mm["miou"] - base["miou"], float(mm["iou"][3] - base["iou"][3])
        verdicts[v] = bool(d_destroyed > KEEP_RULE["min_destroyed_gain"] and d_miou >= -KEEP_RULE["max_miou_drop"])
        gates[v] = bool(mm["miou"] > 0.52 and mm["iou"][3] > 0.30)
        line = (f"  {v:>20}: mIoU {mm['miou']:.4f} ({d_miou:+.4f}) | Destroyed IoU {mm['iou'][3]:.4f} "
                f"({d_destroyed:+.4f}) → {'KEEP' if verdicts[v] else 'not kept'} | Phase Gate {'pass' if gates[v] else 'fail'}")
        if m == f"sam3+{U}":
            vs_step2[v] = {"miou": round(mm["miou"] - STEP2[seed][0], 4),
                           "destroyed_iou": round(float(mm["iou"][3]) - STEP2[seed][1], 4)}
            line += (f" | against step 2 with R11's pieces: mIoU {vs_step2[v]['miou']:+.4f}, "
                     f"Destroyed IoU {vs_step2[v]['destroyed_iou']:+.4f}")
        print(line)
kept_on_all_seeds = {m: all(verdicts[f"R16c_s{seed}_{m}"] for seed in SEEDS) for m in MAPS}
gate_on_all_seeds = {m: all(gates[f"R16c_s{seed}_{m}"] for seed in SEEDS) for m in MAPS}
o1 = ("n/a (smoke test)" if SMOKE_TEST else
      f"SAM 3 + {UNET}'s pieces {'passes' if gate_on_all_seeds[f'sam3+{U}'] else 'does not pass'} the O1 gate on both "
      f"grader seeds; with step 2 (R11's pieces), that is {'two U-Net seeds' if gate_on_all_seeds[f'sam3+{U}'] else 'still one U-Net seed'}.")
print(f"Kept on every seed: {kept_on_all_seeds}; Phase Gate on every seed: {gate_on_all_seeds}")
print(f"\nO1: {o1}")

# 5) Output files
pd.DataFrame(inst_rows).to_csv(OUT_DIR / "instances_val.csv", index=False)

CSV_FIELDS = ["member", "experiment_id", "category", "variable", "value", "miou", "bg_iou",
              "intact_iou", "damaged_iou", "destroyed_iou", "precision", "recall", "f1", "f2",
              "destroyed_precision", "destroyed_recall", "destroyed_f2", "train_time_hrs", "seed", "notes"]

def result_row(exp_id, m, value, notes, seed=None):
    return {
        "member": "Phase C (R16c step 3)", "experiment_id": exp_id, "category": "E-protocol",
        "variable": "find-then-grade", "value": value, "miou": round(m["miou"], 4),
        "bg_iou": round(float(m["iou"][0]), 4), "intact_iou": round(float(m["iou"][1]), 4),
        "damaged_iou": round(float(m["iou"][2]), 4), "destroyed_iou": round(float(m["iou"][3]), 4),
        "precision": round(float(m["precision"].mean()), 4), "recall": round(float(m["recall"].mean()), 4),
        "f1": round(float(m["f1"].mean()), 4), "f2": round(float(m["f2"].mean()), 4),
        "destroyed_precision": round(float(m["precision"][3]), 4),
        "destroyed_recall": round(float(m["recall"][3]), 4), "destroyed_f2": round(float(m["f2"][3]), 4),
        "train_time_hrs": None, "seed": seed, "notes": notes,
    }

def describe(v):
    if v.startswith(UNET):
        return f"{UNET} pixels{' + 8-view TTA' if v.endswith('tta') else ''} (reference)"
    if v.startswith("ceiling"):
        return f"perfect grader{' with rejects' if 'rejects' in v else ''} on {v.split('_')[-1]} outlines (ceiling)"
    seed, rest = v.split("_")[1][1:], v.split("_", 2)[2]
    return (f"step 2's grader, seed {seed}: {rest.replace('_shares', '')} map, "
            f"{RULES['_shares'] if rest.endswith('_shares') else RULES['']}")

with open(OUT_DIR / "D4_R16c_step3.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
    writer.writeheader()
    for v in VARIANTS:
        seed = int(v.split("_")[1][1:]) if v.startswith("R16c_s") else None
        writer.writerow(result_row(v, metrics(cms[v]["all"]), describe(v),
                                   "E-protocol; step 2's graders and biases, R11s7's pieces", seed))

config = {"task": "R16c step 3", "smoke_test": SMOKE_TEST, "val_images": len(val_pairs),
          "sam3_dir": str(SAM3_DIR), "grader_dir": str(GRADER_DIR), "unet": UNET, "unet_checkpoint": str(UNET_CKPT),
          "unet_reproduces_csv": sanity, "expected_unet": EXPECTED_UNET, "graders_reproduce_step2_f1": graders_same,
          "biases": {str(s): dict(zip(LABELS, BIASES[s].tolist())) for s in SEEDS},
          "crop": CROP, "margin": MARGIN, "min_area": MIN_AREA, "seeds": list(SEEDS),
          "validation_minutes": round(VAL_MIN, 1), "missing_sam3_maps": len(missing_sam),
          "true_buildings_macro_f1": {str(s): f for s, f in macro_f1.items()},
          "keep": verdicts, "phase_gate": gates, "vs_step2": vs_step2,
          "kept_on_all_seeds": kept_on_all_seeds, "gate_on_all_seeds": gate_on_all_seeds, "o1": o1}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")

(OUT_DIR / "README.md").write_text("\n".join([
    "# R16c step 3: step 2's graders with R11s7's pieces", "",
    f"{len(val_pairs)} validation images{' (SMOKE TEST)' if SMOKE_TEST else ''}. The finding as in R16: SAM 3 "
    f"`pre_grid2` (E3) plus {UNET}'s building pieces that SAM 3 missed. The grading: step 2's Siamese ResNet-18s "
    f"(seeds {list(SEEDS)}) and their biases, loaded as they were kept; nothing was trained.", "",
    "## Pixel scores (E-protocol)", "",
    "`_shares` rows use R16b's fixed rule instead of the searched biases. `ceiling` rows give each found building "
    "its main true grade (`rejects`: or background).", "",
    as_markdown(overall), "",
    f"{UNET} reproduces its D4 CSV: **{sanity}**. The graders reproduce step 2's macro F1 on the true buildings: "
    f"**{graders_same}** ({macro_f1}).", "",
    f"## Keep rule against {UNET} TTA (searched biases)", "",
    *[f"- {v}: {'KEEP' if k else 'not kept'}; Phase Gate {'pass' if gates[v] else 'fail'}"
      + (f"; against step 2 with R11's pieces: mIoU {vs_step2[v]['miou']:+.4f}, Destroyed IoU "
         f"{vs_step2[v]['destroyed_iou']:+.4f}" if v in vs_step2 else "")
      for v, k in verdicts.items()], "",
    f"Kept on every seed: {kept_on_all_seeds}. Phase Gate on every seed: {gate_on_all_seeds}.", "",
    f"**O1:** {o1}", "",
    "## The graders on the true validation buildings (object level)", "", as_markdown(true_df), "",
    "## Files", "",
    "- `D4_R16c_step3.csv` (D4 format), `R16c_step3_summary.csv` (also per source), `grader_true_buildings.csv`",
    "- `instances_val.csv`: one row per found building (each seed's log-probabilities, R11s7's and the truth's pixels)",
    "- `config.json`", "",
    "## Benchmark use", "",
    "SAM 3 isn't trained, the graders saw DisasterM3's training images only, and R11s7 never trained on benchmark "
    "pictures, so these outputs may answer benchmark questions.",
]), encoding="utf-8")
print(f"\n✓ Wrote {OUT_DIR}: D4_R16c_step3.csv, R16c_step3_summary.csv, grader_true_buildings.csv, "
      "instances_val.csv, config.json, README.md")

                   variant   miou  background_iou  intact_iou  damaged_iou  destroyed_iou  building_iou  destroyed_precision  destroyed_recall
                 R11s7_std 0.5867          0.9756      0.6836       0.4007         0.2869        0.7386               0.5528            0.3736
                 R11s7_tta 0.5913          0.9761      0.6880       0.4081         0.2928        0.7436               0.5661            0.3775
              ceiling_sam3 0.7060          0.9685      0.6243       0.6755         0.5555        0.6304               0.8401            0.6212
        ceiling_sam3+r11s7 0.7692          0.9754      0.7400       0.7383         0.6232        0.7439               0.7297            0.8103
ceiling_rejects_sam3+r11s7 0.8023          0.9817      0.7898       0.7630         0.6749        0.7912               0.8648            0.7545
             R16c_s42_sam3 0.5470          0.9685      0.5769       0.3436         0.2991        0.6304               0.5129            0.4177